## Analyse identified with scPRINT regulatory gene networks in adult Stem Cells

* **Author**: Anna Maguza 
* **Location**: CellZome, GSK company
* **Creation date**: 04.09.2025
* **Last modified date**: 08.09.2025

In this notebook, we perform an extensive analysis of gene networks generated by scPRINT in our previous notebook for adult stem cells. 

We can look at many patterns from the gene networks such as hub genes, gene modules, and gene-gene interactions. Some of them are related to highly variable and differentially expressed genes while others can only be inferred by using the gene networks.

Table of content:

1. Hub genes
2. Network communities
3. Heatmaps visualizing network communities
4. Gene networks
5. Gene set enrichment analysis
6. Comparing with add-on programs from NichCompass analysis

In [1]:
from bengrn import BenGRN
import scanpy as sc
from bengrn.base import train_classifier

from anndata.utils import make_index_unique
from grnndata import utils as grnutils
from grnndata import read_h5ad

from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

from scdataloader import utils as data_utils
import numpy as np

from pyvis import network as pnx
import networkx as nx
import scipy.sparse
import pandas as pd
import gseapy as gp
from gseapy import dotplot

import os
import ssl
import urllib3

%load_ext autoreload
%autoreload 2 


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/am336941/.local/share/uv/python/cpython-3.10.17-macos-aarch64-none/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/am336941/.local/share/uv/python/cpython-3.10.17-macos-aarch64-none/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/am336941/uv_envs/scprint_env230/scprint_env230/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/a

→ connected lamindb: anonymous/test


## Processing GRN Object for Analysis

Let's process the grn_full object similar to the cancer tutorial to enable detailed network analysis.

In [ ]:
# load the generated gene networks from the previous notebook
grn_full = read_h5ad("1_scPRINT/data/grn_sc_adult.h5ad")

In [3]:
# Remove duplicates and process gene symbols
grn_full.var.symbol = make_index_unique(grn_full.var.symbol.astype(str))

# Convert gene ids to symbols for better readability
grn_full.var['ensembl_id'] = grn_full.var.index
grn_full.var.index = grn_full.var.symbol

# Additional step: Remove any remaining duplicates after index assignment
if grn_full.var.index.duplicated().any():
    print(f"Found {grn_full.var.index.duplicated().sum()} duplicate entries after index assignment")
    grn_full = grn_full[:, ~grn_full.var.index.duplicated(keep='first')]
    print("Removed duplicate entries")

print(f"GRN shape: {grn_full.shape}")
print(f"Number of genes: {grn_full.n_vars}")
print(f"Number of cells/observations: {grn_full.n_obs}")

GRN shape: (1030, 3000)
Number of genes: 3000
Number of cells/observations: 1030


In [4]:
grn_full.obs

,cell_index,Source Name,ENA_SAMPLE,BioSD_SAMPLE,organism,disease,organism_part,cell_type,growth_condition,developmental_stage,Material Type,Protocol REF,sample_id,LIBRARY_LAYOUT,cdna_read_size,cell_barcode_size,end_bias,library_construction,sample_barcode_size,umi_barcode_offset,umi_barcode_size,Performer,Assay Name,ENA_EXPERIMENT,ENA_RUN,time,time_unit,n_genes,doublet_scores,predicted_doublets,n_counts,log1p_n_counts,log1p_n_genes,percent_mito,n_counts_mito,percent_ribo,n_counts_ribo,percent_hb,n_counts_hb,percent_top50,cell_passed_qc,qc_cluster,cluster_passed_qc,consensus_fraction,consensus_passed_qc,total_counts,n_genes_by_counts,percent_chrY,XIST-counts,XIST-percentage,sex,S_score,G2M_score,Cell_cycle_phase,Study_name,ArrayExpress_ID,library_preparation_protocol,full_age,age_group,immunophenotype,metadata_cluster,barcode,category,Integrated_05,age,gestational_age,donor_id,passage,batch,sampling_site,celltype,library_construnction_and_layout,cell_states,cellstates_scANVI,confidence_score,gut_region,cell_state,organism_ontology_term_id,cell_type_ontology_term_id,development_stage_ontology_term_id,tissue_ontology_term_id,sex_ontology_term_id,assay_ontology_term_id,self_reported_ethnicity_ontology_term_id,disease_ontology_term_id,cell_culture,nnz,log1p_n_genes_by_counts,log1p_total_counts,pct_counts_in_top_20_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,total_counts_ribo,log1p_total_counts_ribo,pct_counts_ribo,total_counts_hb,log1p_total_counts_hb,pct_counts_hb,outlier,mt_outlier,batches
5a95ec3f-e4cc-4288-b9ad-5266f8707168,TCAGGATAGTAACCCT-1-4918STDY7844899_ERR4010706,T161_IL_cells,ERS4414972,SAMEA6655503,Homo sapiens,normal,terminal ileum,unsorted,primary tissue,child stage,cell,Unknown,T161_IL_cells,PAIRED,98,16,3 prime tag,10xV2,8,16,10,Wellcome Sanger Institute,T161_IL_cells,ERX4012106,ERR4010706,N/A,year,6607,0.194631,False,52314,10.865038,8.796036,10.062316,5264.0,34.937111,18277.0,0.007646,4.0,37.695454,True,8,True,1.0,True,52158.0,6565,0.066904,0,0.000000,male,1.276539,0.389153,S,Elementaite_2021,E-MTAB-8901,10xV2 3 prime tag,4 year,child stage,total cells,0,TCAGGATAGTAACCCT,Unknown,Unknown,4,N/A - not fetal,T161,N/A,4918STDY7844899,unknown,Epithelial,10xV2_PAIRED,Stem cells,Stem cells,0.814317,small intestine,NaN,NCBITaxon:9606,CL:1001566,HsapDv:0000080,UBERON:0002108,PATO:0000384,EFO:0009901,unknown,PATO:0000461,False,6607,8.789660,10.862052,20.575942,5264.0,8.568836,10.092412,18274.0,9.813289,35.035853,4.0,1.609438,0.007669,True,True,"EFO:0009901,unknown,PATO:0000384,T161"
d2d1956a-76d9-47f3-848a-08ab1f2e90c6,CCGTACTCACCATCCT-1-4918STDY7923744_ERR4010707,T182_IL_cells,ERS4414973,SAMEA6655504,Homo sapiens,normal,terminal ileum,unsorted,primary tissue,child stage,cell,Unknown,T182_IL_cells,PAIRED,98,16,3 prime tag,10xV2,8,16,10,Wellcome Sanger Institute,T182_IL_cells,ERX4012107,ERR4010707,N/A,year,5991,0.191710,False,50763,10.834943,8.698347,13.988535,7101.0,37.097886,18832.0,0.000000,0.0,43.374111,True,8,True,1.0,True,50645.0,5949,0.100467,0,0.000000,male,0.833440,1.289970,G2M,Elementaite_2021,E-MTAB-8901,10xV2 3 prime tag,9 year,child stage,total cells,0,CCGTACTCACCATCCT,Epithelial,Stem cells,9,N/A - not fetal,T182,N/A,4918STDY7923744,unknown,Epithelial,10xV2_PAIRED,Stem cells,Stem cells,0.878443,small intestine,NaN,NCBITaxon:9606,CL:1001566,HsapDv:0000080,UBERON:0002108,PATO:0000384,EFO:0009901,unknown,PATO:0000461,False,5991,8.691146,10.832616,25.031099,7101.0,8.868132,14.021127,18827.0,9.843100,37.174450,0.0,0.000000,0.000000,True,True,"EFO:0009901,unknown,PATO:0000384,T182"
443be859-df0d-42c0-a5aa-c23124598e24,GTAGGCCAGGCTCATT-1-4918STDY7923744_ERR4010707,T182_IL_cells,ERS4414973,SAMEA6655504,Homo sapiens,normal,terminal ileum,unsorted,primary tissue,child stage,cell,Unknown,T182_IL_cells,PAIRED,98,16,3 prime tag,10xV2,8,16,10,Wellcome Sanger Institute,T182_IL_cells,ERX4012107,ERR4010707,N/A,year,5797,0.215909,False,47286,10.763991,8.665268,12.944635,6121.0,38.941759,18414.0,0.000000,0.0,43.107897,

In [5]:
network_matrix = grn_full.grn

In [6]:
network_matrix

symbol,TSPAN6,DPM1,FIRRM,FUCA2,CFTR,ANKIB1,KRIT1,BAD,AOC1,TMEM176A,ICA1,DBNDD1,MTMR7,SLC7A2,ARF5,POLDIP2,AK2,FKBP4,KDM1A,NDUFAB1,ST7,HCCS,SLC25A5,HOXA11,LIG3,ACSM3,SPPL2B,FBXL3,PDK2,ITGA3,GDE1,CRLF1,SPATA20,TNFRSF12A,MAP3K9,BAIAP2L1,TTC22,FARP2,USH1C,GGCT,GTF2IRD1,ARSD,PNPLA4,PROM1,NOS2,SLC13A2,TRAPPC6A,TEAD3,NOX1,PSMB1,ADAM22,PLEKHG6,NFIX,MMP25,RHOBTB2,UBE3C,IYD,MLXIPL,ETV7,UQCRC1,CD9,FUZ,PRSS3,HFE,ELOA,MBTD1,UTP18,TTC19,CEP68,ERCC1,PRICKLE3,MVP,PGM3,HEBP1,GPRC5A,CAPN1,MDH1,MTMR11,ZMYND11,DPEP1,CHDH,GLT8D1,ATP2C1,SYT13,ZRANB1,PLEKHB1,SERPINB1,CPS1,SLC45A4,ZDHHC6,GLRX2,DERA,STRAP,NR1H3,NCAPH2,MIPEP,VRK2,SLC39A9,RABEP1,HMGB3,...,CSNK1E,NUDT19,AP1G2,GANC,ZNF891,ALG3,FIS1,MBLAC1,BBIP1,STARD10,IFRD2,ARHGEF28,PHB2,MYL5,TSTD1,CEBPZOS,AP4M1,CKMT1A,ATXN1L,C2CD4D,TMEM185B,HSBP1L1,SRRM5,MT-ATP8,DHFR,PHGR1,BTBD18,TMEM238,CTXN2,ZBED5,ARHGEF38,ZNF593OS,CKMT1B,ASB14,C1orf226,AQP1,C3orf85,RPL36A,ARHGAP8,ARPC4,ATP5PO,nan,PI4KA,EIF6,EIF4EBP3,NME2,LEFTY1,DDOST,P2RY11,TMEM141,CEBPA,PGAM5,ZCCHC3,ADH1C,ABHD14A,SMIM31,FMN1,TMEM150C,TMEM200B,C1orf210,TUG1,ATXN7L3B,PRKDC,LRRC24,DPP3,HMBS,DND1,POLG2,GATC,CNPY2,CUX1,P2RX5-TAX1BP3,nan-1,RNASE4,CEP95,RBM15B,HOXB7,EPPK1,DYNLL2,OTUD7B,NCOA4,SMIM22,SMIM32,CD24,SPMIP1,nan-2,CASTOR2,PPP4R3B,ZNF2,DACH1,MARCKS,GSTT1,SLC9A3,RHOXF1P3,SCO2,nan-3,LRTOMT,nan-4,nan-5,SOD2
symbol,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
TSPAN6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002265,0.0,0.0,0.013398,0.0,0.0,0.0,0.0,0.001904,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.004037,0.000000,0.000000,0.0,0.000000,0.000000,0.028046,0.0,0.000379
DPM1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000842,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000754,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002914,0.0,0.0,0.003531,0.0,0.0,0.0,0.0,0.027473,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000915,0.000000,0.009188,0.000897,0.000000,0.0,0.000000,0.000344,0.027717,0.0,0.000991
FIRRM,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000342,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000449,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000728,0.0,0.0,0.001732,0.0,0.0,0.0,0.0,0.027688,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

## Hub Genes Analysis

We'll analyze hub genes using degree centrality (number of connections) and eigenvector centrality.

In [7]:
# Hub genes based on edge centrality (number of connections / strength of connections)
print("Top 20 hub genes by connection strength:")
top_hubs = grn_full.grn.sum(0).sort_values(ascending=False).head(20)
print(top_hubs)

Top 20 hub genes by connection strength:
symbol
HMGA1       119.876488
MRPL15      104.011520
SENP6        74.496002
nan-4        57.428143
BTBD18       52.086205
GSTT1        51.470516
NDUFB3       49.955280
ZNF593OS     48.956760
ATP5MC1      41.875740
LRRC75A      36.658974
LPP          36.251171
MT-ND6       25.514629
CAMK2A       24.120926
MT-CO1       16.468536
ATG10        14.562601
ZNF593       13.340652
NDUFA8       12.786342
RHOA         12.368204
MT-CYB       12.303049
FTL          11.428805
dtype: float32


In [8]:
# Compute eigenvector centrality by creating a sparse network with top 20 neighbors per gene
TOP = 20

print("Computing centrality measures...")
grnutils.get_centrality(grn_full, TOP, top_k_to_disp=0)

print("Top 20 genes by eigenvector centrality:")
top_centrality = grn_full.var.centrality.sort_values(ascending=False).head(20)
print(top_centrality)

Computing centrality measures...
Top central genes: []
Top 20 genes by eigenvector centrality:
symbol
ATP5MC1     0.288497
SENP6       0.281725
NDUFB3      0.278304
HMGA1       0.276837
MRPL15      0.244644
MT-ND6      0.234026
CAMK2A      0.217194
GSTT1       0.216860
BTBD18      0.211158
MT-CO1      0.207338
LRRC75A     0.202670
MT-CYB      0.202566
MT-ATP8     0.187184
NDUFA8      0.174012
ZNF593      0.172741
FTL         0.170320
ZNF593OS    0.162304
DACH1       0.148604
ATG10       0.148596
nan-4       0.126023
Name: centrality, dtype: float64


## Network Communities Detection

Using Leiden algorithm to find communities in the network, focusing on communities between 20-200 genes.

In [9]:
grn_full = grnutils.compute_cluster(grn_full, resolution=1.5, use="leiden", n_iterations=10, max_comm_size=200)

In [10]:
grn_full.var['cluster_1.5'].value_counts()

cluster_1.5
0     200
1     200
2     200
3     200
4     200
5     200
6     200
7     200
8     200
9     200
10    200
11    200
12    155
13    150
14    124
15     90
16     61
17      5
18      2
19      1
20      1
21      1
22      1
23      1
24      1
25      1
26      1
27      1
28      1
29      1
30      1
31      1
Name: count, dtype: int64

## Create a hierarchically-clustered heatmap

In [ ]:
# delete RPL26L1 
#network_matrix = network_matrix[network_matrix.index != 'PRRC2B']
#grn_full.var['cluster_1.5'] = grn_full.var['cluster_1.5'][grn_full.var['cluster_1.5'] != 'PRRC2B']

In [11]:
# Set high DPI for better quality
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

In [12]:
# Get unique communities and filter out small ones
unique_communities = grn_full.var['cluster_1.5'].unique()
print(f"All communities found: {sorted(unique_communities)}")

All communities found: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '4', '5', '6', '7', '8', '9']


In [15]:
# Filter communities with at least 100 genes
valid_communities = []
community_gene_counts = {}

for community in unique_communities:
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    community_gene_counts[community] = len(community_genes)
    if len(community_genes) >= 100:
        valid_communities.append(community)

In [16]:
# Select top N genes from each valid community
selected_genes = []
community_labels = []
top_n = 200 
for community in sorted(valid_communities):
    # Get genes in this community
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    
    # Calculate gene strengths (total interaction strength) for genes in this community
    gene_strengths = []
    for gene in community_genes:
        if gene in network_matrix.index:
            # Calculate total absolute interaction strength for this gene
            gene_total_strength = network_matrix.loc[gene, :].abs().sum()
            gene_strengths.append((gene, gene_total_strength))
        else:
            gene_strengths.append((gene, 0))  # Missing genes get 0 strength
    
    # Sort by strength (descending) and select top N
    gene_strengths.sort(key=lambda x: x[1], reverse=True)
    selected_community_genes = [gene for gene, strength in gene_strengths[:top_n]]
    
    selected_genes.extend(selected_community_genes)
    community_labels.extend([community] * len(selected_community_genes))

In [17]:
# Check which genes exist in network_matrix
missing_genes = []
valid_genes = []

for gene in selected_genes:
    if gene in network_matrix.index and gene in network_matrix.columns:
        valid_genes.append(gene)
    else:
        missing_genes.append(gene)

print(f"Valid genes in network_matrix: {len(valid_genes)}")
print(f"Missing genes: {len(missing_genes)}")

Valid genes in network_matrix: 2829
Missing genes: 0


In [18]:
# Filter network matrix to selected genes
filtered_matrix = network_matrix.loc[valid_genes, valid_genes]

# Update community labels for valid genes only
valid_community_labels = []
for gene in valid_genes:
    original_index = selected_genes.index(gene)
    valid_community_labels.append(community_labels[original_index])

print(f"\nFinal matrix shape: {filtered_matrix.shape}")


Final matrix shape: (2829, 2829)


In [19]:
# Calculate community-level interaction matrices for clustering
community_interaction_matrix = pd.DataFrame(index=sorted(valid_communities), 
                                           columns=sorted(valid_communities))

for comm1 in sorted(valid_communities):
    for comm2 in sorted(valid_communities):
        # Get genes from each community
        genes_comm1 = [gene for i, gene in enumerate(valid_genes) 
                      if valid_community_labels[i] == comm1]
        genes_comm2 = [gene for i, gene in enumerate(valid_genes) 
                      if valid_community_labels[i] == comm2]
        
        # Calculate mean interaction between communities
        if genes_comm1 and genes_comm2:
            interaction_submatrix = filtered_matrix.loc[genes_comm1, genes_comm2]
            mean_interaction = interaction_submatrix.values.mean()
            community_interaction_matrix.loc[comm1, comm2] = mean_interaction
        else:
            community_interaction_matrix.loc[comm1, comm2] = 0

# Convert to numeric
community_interaction_matrix = community_interaction_matrix.astype(float)

In [20]:
community_linkage = linkage(pdist(community_interaction_matrix, metric='euclidean'), method='ward')

In [21]:
# Get community ordering from clustering
community_dendro = dendrogram(community_linkage, no_plot=True)
community_order = [sorted(valid_communities)[i] for i in community_dendro['leaves']]

print(f"Community clustering order: {community_order}")

Community clustering order: ['1', '11', '3', '14', '8', '0', '6', '10', '12', '13', '9', '5', '2', '4', '7']


In [22]:
# Reorder genes based on community clustering
reordered_genes = []
reordered_community_labels = []

for community in community_order:
    community_genes = [gene for i, gene in enumerate(valid_genes) 
                      if valid_community_labels[i] == community]
    reordered_genes.extend(community_genes)
    reordered_community_labels.extend([community] * len(community_genes))

In [23]:
# Reorder matrix
gene_to_index = {gene: i for i, gene in enumerate(valid_genes)}
new_indices = [gene_to_index[gene] for gene in reordered_genes]
reordered_matrix = filtered_matrix.iloc[new_indices, new_indices]

In [24]:
# Community colors
community_colors = plt.cm.tab20(np.linspace(0, 1, len(valid_communities)))
community_color_map = {comm: color for comm, color in zip(sorted(valid_communities), community_colors)}

In [ ]:
# Create figure with gridspec for better control
fig = plt.figure(figsize=(24, 20))
gs = GridSpec(4, 4, figure=fig, height_ratios=[1.2, 0.05, 3, 0.3], width_ratios=[0.3, 0.05, 3, 0.8])

# Top dendrogram - showing community clustering
ax_dendro_top = fig.add_subplot(gs[0, 2])
dendro_top = dendrogram(community_linkage, ax=ax_dendro_top, orientation='top', 
                       labels=[f'Comm {c}' for c in sorted(valid_communities)],
                       color_threshold=0, leaf_font_size=10)
ax_dendro_top.set_title('Community Similarity Dendrogram', fontsize=12, fontweight='bold', pad=10)
ax_dendro_top.set_xticks([])
ax_dendro_top.tick_params(axis='y', labelsize=8)

# Left dendrogram - same community clustering  
ax_dendro_left = fig.add_subplot(gs[2, 0])
dendro_left = dendrogram(community_linkage, ax=ax_dendro_left, orientation='left',
                        labels=[f'Comm {c}' for c in sorted(valid_communities)],
                        color_threshold=0, leaf_font_size=10)
ax_dendro_left.set_ylabel('Community Similarity Dendrogram', fontsize=12, fontweight='bold')
ax_dendro_left.set_yticks([])
ax_dendro_left.tick_params(axis='x', labelsize=8)

# Main heatmap
ax_heatmap = fig.add_subplot(gs[2, 2])
vmax = max(abs(reordered_matrix.min().min()), abs(reordered_matrix.max().max()))
vmin = -vmax

im = ax_heatmap.imshow(reordered_matrix, cmap='RdBu_r', aspect='auto', vmin=vmin, vmax=vmax)

# Set gene labels on axes with smaller font
ax_heatmap.set_xticks(range(len(reordered_genes)))
ax_heatmap.set_yticks(range(len(reordered_genes)))
ax_heatmap.set_xticklabels(reordered_genes, rotation=90, ha='center', fontsize=1)
ax_heatmap.set_yticklabels(reordered_genes, fontsize=1)

# Remove tick marks but keep labels
ax_heatmap.tick_params(axis='both', which='both', length=0, width=0, pad=1)

# Community color bar on the left
ax_community = fig.add_subplot(gs[2, 1])
community_colors_reordered = [community_color_map[label] for label in reordered_community_labels]
# Create proper color array for imshow - need to ensure it's in the right format
community_array = np.array(community_colors_reordered).reshape(-1, 1, 4)  # height x width x RGBA
ax_community.imshow(community_array, aspect='auto')
ax_community.set_xticks([])
ax_community.set_yticks([])
ax_community.set_ylabel('Communities', fontsize=12, rotation=90, fontweight='bold')

# Community color bar on the top
ax_community_top = fig.add_subplot(gs[1, 2])
# Create proper color array for imshow - need to ensure it's in the right format
community_array_top = np.array(community_colors_reordered).reshape(1, -1, 4)  # height x width x RGBA
ax_community_top.imshow(community_array_top, aspect='auto')
ax_community_top.set_xticks([])
ax_community_top.set_yticks([])
ax_community_top.set_xlabel('Communities', fontsize=12, fontweight='bold')

# Add community boundaries
current_pos = 0
for community in community_order:
    community_size = reordered_community_labels.count(community)
    if community_size > 0:
        # Add thin white lines to separate communities
        if current_pos > 0:
            ax_heatmap.axhline(y=current_pos-0.5, color='white', linewidth=1)
            ax_heatmap.axvline(x=current_pos-0.5, color='white', linewidth=1)
        current_pos += community_size

# Colorbar
gs = GridSpec(4, 4, figure=fig, height_ratios=[1.2, 0.05, 3, 0.3], width_ratios=[0.3, 0.05, 3, 0.1])
ax_cbar = fig.add_subplot(gs[2, 3])
cbar = plt.colorbar(im, cax=ax_cbar)
cbar.set_label('Gene Interaction Strength', fontsize=12, fontweight='bold')
cbar.ax.tick_params(labelsize=10)

# Add legend
legend_patches = []
for community in community_order:  # Use clustered order
    color = community_color_map[community]
    count = reordered_community_labels.count(community)
    legend_patches.append(mpatches.Patch(color=color, label=f'Community {community} ({count} genes)'))

ax_legend = fig.add_subplot(gs[3, :])
ax_legend.legend(handles=legend_patches, loc='center', ncol=min(5, len(valid_communities)), fontsize=11)
ax_legend.axis('off')

plt.savefig("1_scPRINT/adultSC_figures/community_heatmaps/all_genes_community_clustering_results.png", bbox_inches='tight', dpi=300)

## Create a hierarchically-clustered heatmap for each community

In [ ]:
# Set high DPI for better quality
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# Get unique communities
unique_communities = grn_full.var['cluster_1.5'].unique()
print(f"All communities found: {sorted(unique_communities)}")

# Filter communities with at least 40 genes
valid_communities_for_heatmaps = []
community_gene_counts = {}

for community in unique_communities:
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    community_gene_counts[community] = len(community_genes)
    if len(community_genes) >= 40:
        valid_communities_for_heatmaps.append(community)

# Create directory for saving plots if it doesn't exist
output_dir = "community_heatmaps"
os.makedirs(output_dir, exist_ok=True)

# Create individual heatmaps for each valid community
for community in sorted(valid_communities_for_heatmaps):
    
    # Get all genes in this community
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == community].index.tolist()
    
    # Filter to genes that exist in network_matrix
    valid_community_genes = [gene for gene in community_genes 
                            if gene in network_matrix.index and gene in network_matrix.columns]
    
    if len(valid_community_genes) < 5:
        print(f"  SKIPPED: Too few valid genes ({len(valid_community_genes)})")
        continue
    
    # Extract submatrix for this community
    community_matrix = network_matrix.loc[valid_community_genes, valid_community_genes]
    
    # Calculate matrix statistics
    non_zero_interactions = (community_matrix != 0).sum().sum()
    total_interactions = community_matrix.size
    matrix_density = non_zero_interactions / total_interactions
    
    # Determine figure size based on number of genes
    base_size = max(8, min(20, len(valid_community_genes) * 0.3))
    figsize = (base_size, base_size)
    
    # Create clustermap with dendrogram
    try:
        # Calculate gene label font size based on number of genes
        if len(valid_community_genes) <= 50:
            label_fontsize = 6
        elif len(valid_community_genes) <= 100:
            label_fontsize = 4
        else:
            label_fontsize = 3
        
        # Calculate clustering once and apply to both axes
        linkage_matrix = linkage(pdist(community_matrix, metric='euclidean'), method='ward')
        
        # Create clustermap with same clustering for rows and columns
        g = sns.clustermap(community_matrix,
                          row_linkage=linkage_matrix,
                          col_linkage=linkage_matrix,  # Use same linkage for both axes
                          figsize=figsize,
                          cmap='RdBu_r',
                          center=0,
                          square=True,
                          linewidths=0.01,
                          cbar_kws={"shrink": 0.8, "label": "Interaction Strength"},
                          xticklabels=True,
                          yticklabels=True)
        
        # Get the clustered gene order
        row_order = g.dendrogram_row.reordered_ind
        clustered_genes = [valid_community_genes[i] for i in row_order]
        
        # Ensure both axes have the same gene labels in the same order
        g.ax_heatmap.set_xticklabels(clustered_genes, 
                                    rotation=90, ha='center', fontsize=label_fontsize)
        g.ax_heatmap.set_yticklabels(clustered_genes, 
                                    rotation=0, fontsize=label_fontsize)
        
        # Remove tick marks but keep labels
        g.ax_heatmap.tick_params(axis='both', which='both', length=0, width=0, pad=1)
        
        # Add title
        g.fig.suptitle(f'Community {community} - Internal Gene Interactions\n'
                      f'{len(valid_community_genes)} genes, {matrix_density:.1%} density\n'
                      f'Range: [{community_matrix.min().min():.3f}, {community_matrix.max().max():.3f}]',
                      fontsize=14, fontweight='bold', y=0.98)
        
        # Save the plot
        filename = f"community{community}.png"
        filepath = os.path.join(output_dir, filename)
        g.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  ✓ Saved: {filepath}")
        
        # Close the figure to free memory
        plt.close(g.fig)

    except Exception as e:
        print(f"  ✗ Error creating heatmap for community {community}: {str(e)}")
        continue

## Gene networks

In [ ]:
grn_full.var['cluster_1.5']

In [31]:
community_counts = grn_full.var['cluster_1.5'].value_counts()

In [ ]:
# delete communities with less than 40 genes
min_genes = 40
valid_communities = community_counts[community_counts >= min_genes].index

for community in valid_communities:
    selected_community = community
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == selected_community].index.tolist()
    
    # Create a subgraph for this specific community
    G = grn_full.plot_subgraph(community_genes, only=100, interactive=False)
    
    plt.figure(figsize=(12, 10))

    # Use a layout algorithm for better visualization
    pos = nx.spring_layout(G, k=1, iterations=50)

    # Draw the network
    nx.draw(G, pos,
            with_labels=True,
            node_color='lightblue',
            node_size=500,
            font_size=8,
            font_weight='bold',
            arrows=True,
            edge_color='gray',
            alpha=0.7)

    plt.title(f'Gene Regulatory Network - Community {selected_community}',
              fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save as PNG
    plt.savefig(f"1_scPRINT/adultSC_figures/networks/grn_community_{selected_community}.png",
                dpi=300, bbox_inches='tight')
    plt.close()

## Gene set enrichment analysis

In [33]:
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Set environment variables to disable SSL verification
os.environ['PYTHONHTTPSVERIFY'] = '0'
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['SSL_VERIFY'] = 'false'

# Configure SSL context
ssl._create_default_https_context = ssl._create_unverified_context

# Configure requests to not verify SSL
import requests.packages.urllib3.util.ssl_
requests.packages.urllib3.util.ssl_.DEFAULT_CIPHERS = 'ALL'

# Monkey patch requests to disable SSL verification
original_request = requests.Session.request
def patched_request(self, method, url, **kwargs):
    kwargs['verify'] = False
    return original_request(self, method, url, **kwargs)
requests.Session.request = patched_request


In [ ]:
# delete communities with less than 40 genes
min_genes = 40
valid_communities = community_counts[community_counts >= min_genes].index

for community in valid_communities:
    selected_community = community
    grn_full.get(grn_full.var[grn_full.var['cluster_1.5']== selected_community].index.tolist()).grn.sum(0).sort_values(ascending=False)
    community_genes = grn_full.var[grn_full.var['cluster_1.5'] == selected_community].index.tolist()
    
    try:
        # Use community-specific genes for enrichment analysis
        enr = gp.enrichr(gene_list=community_genes,  # Changed from list(G.nodes) to community_genes
                        gene_sets=['KEGG_2021_Human', 'MSigDB_Hallmark_2020', 'Reactome_2022', 
                        'Tabula_Sapiens', 'WikiPathway_2023_Human', 'TF_Perturbations_Followed_by_Expression', 
                        'Reactome', 'PPI_Hub_Proteins', 'OMIM_Disease', 'GO_Molecular_Function_2023'],
                        organism='Human',
                        background=grn_full.var.symbol.tolist())

        enr.res2d.Term = enr.res2d.Term.str.split("(").str[0].str[:50]
        ax = dotplot(enr.res2d.replace([np.inf, -np.inf], 0).dropna(),
                    column="Adjusted P-value",
                    title=f'adult Stem Cells Community {selected_community}',
                    cmap=plt.cm.viridis,
                    size=1,
                    top_term=15,
                    figsize=(7,7), 
                    cutoff=0.25)
        
        # Reduce title and axis font sizes
        plt.title(f'Adult Stem Cells Community {selected_community}', fontsize=10)
        plt.xticks(fontsize=7)
        plt.yticks(fontsize=7)
        
        plt.tight_layout()
        plt.savefig(f"1_scPRINT/adultSC_figures/enrichment_plots/enrichment_plot_{selected_community}.png",
                    dpi=300, bbox_inches='tight')
        plt.show()
        
    except ValueError as e:
        if "No enrich terms when cutoff" in str(e):
            print(f"Skipping community {selected_community}: No enriched terms found at cutoff=0.25")
        else:
            print(f"Skipping community {selected_community}: {str(e)}")
        continue
    except Exception as e:
        print(f"Skipping community {selected_community}: Unexpected error - {str(e)}")
        continue

## Comparing with add-on programs from NichCompass analysis

In [ ]:
# Find interesting genes to focus on (e.g., stem cell markers, developmental genes)
stem_cell_markers = [
    'INSM1', 'HES6', 'PROX1', 
    'SOX4','SPIB', 'SOX9', 
    "FOXA3","IRF8", "HMGP2", "L1TD1", "LGR5"] # genes from add-on programs

print("Processing stem cell markers:")
print(f"Total markers to check: {len(stem_cell_markers)}")

# Create directory for saving plots if it doesn't exist
marker_plots_dir = "1_scPRINT/adultSC_figures/marker_networks"
os.makedirs(marker_plots_dir, exist_ok=True)

# Process each marker gene individually
for focus_gene in stem_cell_markers:
    if focus_gene not in grn_full.var.index:
        print(f"Skipping {focus_gene}: Gene not found in network")
        continue
    
    try:
        print(f"Analyzing network around {focus_gene}...")
        
        # Create a subgraph for this specific gene
        G_focus = grn_full.plot_subgraph(focus_gene, only=50, max_genes=30, interactive=False)
        
        if len(G_focus.nodes) == 0:
            print(f"Skipping {focus_gene}: No network connections found")
            continue
        
        plt.figure(figsize=(12, 10))

        # Use a layout algorithm for better visualization
        pos = nx.spring_layout(G_focus, k=1, iterations=50)

        # Draw the network
        nx.draw(G_focus, pos,
                with_labels=True,
                node_color='lightblue',
                node_size=500,
                font_size=8,
                font_weight='bold',
                arrows=True,
                edge_color='gray',
                alpha=0.7)

        # Highlight the focus gene if it exists in the network
        if focus_gene in G_focus.nodes:
            pos_focus = pos[focus_gene]
            nx.draw_networkx_nodes(G_focus, pos, nodelist=[focus_gene], 
                                 node_color='red', node_size=800, alpha=0.9)

        plt.title(f'Gene Regulatory Network - {focus_gene}',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()

        # Save as PNG
        filename = f"grn_marker_{focus_gene}_network.png"
        filepath = os.path.join(marker_plots_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved: {filepath}")
        plt.close()
        
    except Exception as e:
        print(f"Error processing {focus_gene}: {str(e)}")
        continue

print(f"\nCompleted processing stem cell markers")

## Analyse all TFs

In [ ]:
tfs = grn_full.var.index[grn_full.var["TFs"]].tolist()
tfs

In [ ]:
tfs_set = set(tfs)

# Filter the centrality scores to only include TFs
tf_centrality = grn_full.var.centrality[grn_full.var.index.isin(tfs_set)].sort_values(ascending=False)

# Display top 20 TFs by centrality
print("Top 20 TFs by centrality score:")
print(tf_centrality.head(20))

# You can also create a bar plot of the top TFs
plt.figure(figsize=(12, 6))
tf_centrality.head(20).plot(kind='bar')
plt.title('Top 20 TFs by Centrality Score')
plt.xlabel('Transcription Factors')
plt.ylabel('Centrality Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig("1_scPRINT/adultSC_figures/top_20_TFs_centrality.png", 
            dpi=300, bbox_inches='tight')
plt.close()

Top 20 TFs by centrality score:
symbol
HMGA1      2.768369e-01
DACH1      1.486040e-01
NFIA       4.306005e-03
NFXL1      9.563627e-04
NFIB       3.202276e-11
NR1D2      1.521721e-11
FOXP1      1.316776e-11
FOXK1      5.123641e-14
OSR2       5.123641e-14
CDX2       5.123641e-14
ZNF219     5.123641e-14
ZSCAN21    5.123641e-14
ZNF606     5.123641e-14
MESP1      5.123641e-14
TET2       5.123641e-14
ZNF444     5.123641e-14
ZNF704     5.123641e-14
CXXC4      5.123641e-14
ZBTB5      5.123641e-14
ZNF608     5.123641e-14
Name: centrality, dtype: float64


In [63]:
tf_list = list(tf_centrality.index)[:5]
tf_list

['HMGA1', 'DACH1', 'NFIA', 'NFXL1', 'NFIB']

In [61]:
grn_full.var

,gene_symbol,gene_id-query,uid,symbol,biotype,organism_id,mt,ribo,hb,organism,ensembl_gene_id,n_cells_by_counts,mean_counts,log1p_mean_counts,pct_dropout_by_counts,total_counts,log1p_total_counts,highly_variable,highly_variable_rank,means,variances,variances_norm,highly_variable_nbatches,TFs,ensembl_id,centrality,cluster_1.5
symbol,,,,,,,,,,,,,,,,,,,,,,,,,,,
TSPAN6,TSPAN6,ENSG00000000003,1uTi9dROoaN5,TSPAN6,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000000003,618,0.035753,0.035128,97.621613,929.0,6.835185,False,1306.0,0.035753,0.075426,0.713581,21,False,ENSG00000000003,5.123641e-14,9
DPM1,DPM1,ENSG00000000419,4eC1wUNJAO2s,DPM1,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000000419,1990,0.093865,0.089718,92.341441,2439.0,7.799753,False,1658.0,0.093865,0.138862,0.729487,13,False,ENSG00000000419,5.123641e-14,0
FIRRM,FIRRM,ENSG00000000460,3wEqb6eOTZzC,FIRRM,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000000460,481,0.021398,0.021172,98.148861,556.0,6.322565,False,1863.0,0.021398,0.027406,0.768021,11,False,ENSG00000000460,5.123641e-14,0
FUCA2,FUCA2,ENSG00000001036,5kgAeF5qhzAi,FUCA2,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000001036,1092,0.054072,0.052661,95.797414,1405.0,7.248504,False,1580.0,0.054072,0.088097,0.693110,9,False,ENSG00000001036,5.123641e-14,3
CFTR,CFTR,ENSG00000001626,1OL6oMCceGiG,CFTR,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000001626,158,0.006735,0.006712,99.391933,175.0,5.170484,False,2160.0,0.006735,0.008152,0.593471,7,False,ENSG00000001626,5.123641e-14,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
nan-3,ENSG00000284526,ENSG00000284526,3I5L12hixNgK,nan-3,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000284526,2422,0.171913,0.158638,90.678879,4467.0,8.404696,True,909.0,0.171913,0.432477,0.868811,16,False,ENSG00000284526,4.969932e-12,0
LRTOMT,LRTOMT,ENSG00000284922,xcz8KWgw5J1x,LRTOMT,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000284922,1822,0.117149,0.110780,92.987993,3044.0,8.021256,True,517.0,0.117149,0.270000,0.871724,21,False,ENSG00000284922,1.668602e-02,12
nan-4,ENSG00000285444,ENSG00000285444,3n9sSEkZVNQf,nan-4,protein_coding,105,False,False,False,NCBITaxon:9606,ENSG00000285444,186,0.009006,0.008965,99.284175,234.0,5.459586,False,1041.0,0.009006,0.013389,0.754798,15,False,ENSG00000285444,1.260233e-01,11


In [ ]:
# Find interesting genes to focus on (e.g., stem cell markers, developmental genes)
tf_list = list(tf_centrality.index)[:5]
tf_list

# Create directory for saving plots if it doesn't exist
marker_plots_dir = "1_scPRINT/adultSC_figures/TF_networks"
os.makedirs(marker_plots_dir, exist_ok=True)

# Process each TF gene individually
for focus_gene in tf_list:
    if focus_gene not in grn_full.var.index:
        print(f"Skipping {focus_gene}: Gene not found in network")
        continue
    
    try:
        print(f"Analyzing network around {focus_gene}...")
        
        # Create a subgraph for this specific gene
        G_focus = grn_full.plot_subgraph(focus_gene, only=50, max_genes=30, interactive=False)
        
        if len(G_focus.nodes) == 0:
            print(f"Skipping {focus_gene}: No network connections found")
            continue
        
        plt.figure(figsize=(12, 10))

        # Use a layout algorithm for better visualization
        pos = nx.spring_layout(G_focus, k=1, iterations=50)

        # Draw the network
        nx.draw(G_focus, pos,
                with_labels=True,
                node_color='lightblue',
                node_size=500,
                font_size=8,
                font_weight='bold',
                arrows=True,
                edge_color='gray',
                alpha=0.7)

        # Highlight the focus gene if it exists in the network
        if focus_gene in G_focus.nodes:
            pos_focus = pos[focus_gene]
            nx.draw_networkx_nodes(G_focus, pos, nodelist=[focus_gene], 
                                 node_color='red', node_size=800, alpha=0.9)

        plt.title(f'Gene Regulatory Network - {focus_gene}',
                  fontsize=14, fontweight='bold')
        plt.tight_layout()

        # Save as PNG
        filename = f"grn_marker_{focus_gene}_network.png"
        filepath = os.path.join(marker_plots_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        
    except Exception as e:
        print(f"Error processing {focus_gene}: {str(e)}")
        continue